# 05 — Generate final tables and figures
Aggregates every JSON in `results/metrics/` into the main and ablation tables, then renders the four poster figures.


## Colab setup
Verify GPU, mount Drive, clone the repo, install requirements, and create the project tree on Drive.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Mount Drive (skipped automatically when not on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cs4782_patchtst_project'
    IN_COLAB = True
except Exception:
    PROJECT_ROOT = '.'
    IN_COLAB = False
print('PROJECT_ROOT =', PROJECT_ROOT, ' IN_COLAB =', IN_COLAB)


In [ ]:
# 3. Clone repo (Colab only). Update REPO_URL in scripts/build_notebooks.py
# and re-run that script to regenerate notebooks if the URL changes.
REPO_URL = 'https://github.com/Ash1R/ATSW64W-experiments.git'
if IN_COLAB:
    import os, subprocess
    os.chdir('/content')
    # Derive the clone directory from the URL's basename so it matches the repo name.
    REPO_DIRNAME = REPO_URL.rstrip('/').rsplit('/', 1)[-1]
    if REPO_DIRNAME.endswith('.git'):
        REPO_DIRNAME = REPO_DIRNAME[:-4]
    if not os.path.isdir(f'/content/{REPO_DIRNAME}'):
        # check=True so a bad URL fails loudly here instead of crashing the next chdir.
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(f'/content/{REPO_DIRNAME}')
    subprocess.run(['git', 'pull'], check=False)
print('cwd =', __import__('os').getcwd())


In [ ]:
# 4. Install requirements (best-effort; resolved relative to the repo root
# regardless of where the kernel started, so headless `nbconvert` runs work).
import subprocess, sys, os
_req_dir = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_req_dir, 'requirements.txt')):
        break
    _req_dir = os.path.dirname(_req_dir)
_req = os.path.join(_req_dir, 'requirements.txt')
if os.path.isfile(_req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req], check=False)
else:
    print('skipping pip install — requirements.txt not found from', os.getcwd())


In [ ]:
# 5. Make the project's `code/` directory importable.
# We add `code/` itself to sys.path (not the repo root) because the
# stdlib already ships a module named `code` that the IPython kernel
# imports before this cell runs — shadowing that cleanly is messy.
# This way every import is `from utils...`, `from data...`, `from models...`.
import sys, os
# When run with `jupyter nbconvert --execute`, the kernel's cwd is
# the notebook's directory (`notebooks/`), so locate the repo root
# by walking up until we find `code/`. On Colab we already chdir'd
# into the cloned repo above.
REPO_DIR = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(REPO_DIR, 'code')):
        break
    REPO_DIR = os.path.dirname(REPO_DIR)
CODE_DIR = os.path.join(REPO_DIR, 'code')
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f'could not locate code/ from {os.getcwd()}')
os.chdir(REPO_DIR)
for p in (REPO_DIR, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
print('REPO_DIR =', REPO_DIR)
from utils.colab import ensure_dirs
subdirs = ensure_dirs(PROJECT_ROOT)
for k, v in subdirs.items():
    print(f'{k:>12}  {v}')


In [ ]:
import os, glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'results')
TABLES = os.path.join(OUTPUT_DIR, 'tables'); os.makedirs(TABLES, exist_ok=True)
FIGURES = os.path.join(OUTPUT_DIR, 'figures'); os.makedirs(FIGURES, exist_ok=True)

# DLinear is no longer trained locally — we cite the PatchTST paper (Nie et al.,
# ICLR 2023) Table 3 instead. TODO: verify values against your copy of Table 3
# (multivariate, normalized test MSE, look-back 336).
PAPER_DLINEAR_MSE_NORM = {
    ('weather',     96):  0.196,
    ('weather',    336):  0.283,
    ('electricity', 96):  0.140,
    ('electricity',336):  0.169,
    ('traffic',     96):  0.410,
    ('traffic',    336):  0.436,
}

rows = []
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'metrics', '*.json'))):
    s = json.load(open(path))
    cfg = s.get('config', {})
    if cfg.get('model') == 'dlinear':
        continue   # stale local DLinear runs no longer used; see PAPER_DLINEAR_MSE_NORM
    rows.append({
        'run_name': s['run_name'],
        'dataset': cfg.get('dataset'),
        'model': cfg.get('model'),
        'seq_len': cfg.get('seq_len'),
        'pred_len': cfg.get('pred_len'),
        'patch_len': cfg.get('patch_len'),
        'stride': cfg.get('stride'),
        'mse_inv': s.get('test_mse_inverse'),
        'mae_inv': s.get('test_mae_inverse'),
        'mse_norm': s.get('test_mse_normalized'),
        'mae_norm': s.get('test_mae_normalized'),
        'num_patches': s.get('num_patches'),
        'attention_pairs': s.get('attention_pairs'),
        'avg_epoch_s': s.get('avg_epoch_seconds'),
        'num_params': s.get('num_params'),
    })
df = pd.DataFrame(rows)
df.to_csv(os.path.join(TABLES, 'all_runs.csv'), index=False)
df


## Main results table — naive vs PatchTST (DLinear baseline cited from paper)


In [ ]:
PAPER_PATCHTST_MSE_NORM = {('weather', 96): 0.152, ('weather', 336): 0.249,
                           ('electricity', 96): 0.130, ('electricity', 336): 0.167}

# Use NORMALIZED MSE for the cross-dataset main table — the paper's metric.
# `mse_inv` is in raw physical units and is not comparable between datasets.
main = (df[df['model'].isin(['naive', 'patchtst']) &
          (df['patch_len'].fillna(16) == 16) & (df['stride'].fillna(8) == 8)]
        .pivot_table(index=['dataset', 'pred_len'], columns='model',
                     values='mse_norm', aggfunc='min')
        .reset_index())
main['paper_dlinear']  = main.apply(lambda r: PAPER_DLINEAR_MSE_NORM.get((r['dataset'], r['pred_len'])), axis=1)
main['paper_patchtst'] = main.apply(lambda r: PAPER_PATCHTST_MSE_NORM.get((r['dataset'], r['pred_len'])), axis=1)
main = main[['dataset', 'pred_len', 'naive', 'paper_dlinear', 'patchtst', 'paper_patchtst']]
main.to_csv(os.path.join(TABLES, 'main_results.csv'), index=False)
with open(os.path.join(TABLES, 'main_results_latex.txt'), 'w') as f:
    f.write(main.to_latex(index=False, float_format='%.4f'))
main


## Ablation table


In [ ]:
abl_rows = df[(df['dataset'] == 'weather') & (df['pred_len'] == 96)
              & df['model'].isin(['patchtst'])
              & df['run_name'].str.contains('NOPATCH|P\\d+S\\d+|T96_L\\d+$', regex=True)]
abl = abl_rows[['run_name', 'patch_len', 'stride', 'seq_len', 'num_patches',
                'attention_pairs', 'mse_inv', 'mae_inv', 'avg_epoch_s']].copy()
abl.to_csv(os.path.join(TABLES, 'ablation_results.csv'), index=False)
with open(os.path.join(TABLES, 'ablation_results_latex.txt'), 'w') as f:
    f.write(abl.to_latex(index=False, float_format='%.4f'))
abl


## Figure 1 — main results bar chart


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, T in zip(axes, [96, 336]):
    sub = main[main['pred_len'] == T].set_index('dataset')
    sub[['naive', 'paper_dlinear', 'patchtst']].plot.bar(ax=ax)
    ax.set_title(f'Test MSE @ T={T}  (DLinear from PatchTST paper Table 3)')
    ax.set_ylabel('MSE (normalized)')
    ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, 'main_results_bar.png'), dpi=150)
plt.show()


## Figure 2 — patching efficiency (token count vs MSE)


In [ ]:
eff = abl.dropna(subset=['num_patches']).copy()
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(eff['num_patches'], eff['mse_inv'], s=80)
for _, r in eff.iterrows():
    ax.annotate(f"P={int(r['patch_len'])}/S={int(r['stride'])}",
                (r['num_patches'], r['mse_inv']),
                textcoords='offset points', xytext=(6, 4))
ax.set_xscale('log')
ax.set_xlabel('# tokens N (log)')
ax.set_ylabel('Test MSE (inverse-scaled)')
ax.set_title('Patching efficiency on Weather T=96')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, 'patching_efficiency.png'), dpi=150)
plt.show()


## Figure 3 — patch-size and look-back sweeps


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
psweep = abl[abl['run_name'].str.contains('P\\d+S\\d+', regex=True)].sort_values('patch_len')
axes[0].plot(psweep['patch_len'], psweep['mse_inv'], marker='o')
axes[0].set_xlabel('patch length P'); axes[0].set_ylabel('MSE'); axes[0].set_title('Patch-size sweep')
axes[0].grid(True, alpha=0.3)
lsweep = abl[abl['run_name'].str.contains('T96_L\\d+$', regex=True)].sort_values('seq_len')
axes[1].plot(lsweep['seq_len'], lsweep['mse_inv'], marker='s', color='tab:orange')
axes[1].set_xlabel('look-back length L'); axes[1].set_ylabel('MSE'); axes[1].set_title('Look-back sweep')
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, 'patch_size_sweep.png'), dpi=150)
fig.savefig(os.path.join(FIGURES, 'lookback_sweep.png'), dpi=150)
plt.show()


## Figure 4 — example prediction


In [ ]:
import shutil
src = os.path.join(FIGURES, 'weather_patchtst_L336_T96_P16S8_seed42_prediction_plot.png')
dst = os.path.join(FIGURES, 'example_predictions.png')
if os.path.exists(src):
    shutil.copyfile(src, dst)
    print('copied to', dst)
else:
    print('expected', src, 'not found — run notebook 02 first')
